Day 7 — Full Training Loop
Brain Tumour Detection Project
==============================
Topics covered:
  1.  Epoch / batch / iteration — the arithmetic of training
  2.  gradient accumulation — why zero_grad() is mandatory
  3.  Weighted loss accumulation — correct epoch averaging
  4.  train_one_epoch() — the core training function
  5.  evaluate() — validation with no_grad and eval mode
  6.  Checkpointing — deepcopy, what to save, why val loss
  7.  Scheduler placement — per-epoch not per-batch
  8.  Full training run — 50 epochs with live metrics
  9.  Loss curve interpretation — reading overfitting
  10. Final test evaluation + verification checklist


In [1]:
import os, time, copy
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from collections import Counter
from PIL import Image
 
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(42); np.random.seed(42)

In [2]:
# ── CONFIG ──────────────────────────────────────────────
DATA_DIR     = "data/brain_tumour"   # ← your Kaggle dataset path
IMG_SIZE     = 128
BATCH_SIZE   = 32
EPOCHS       = 50
LR           = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 0
CKPT_PATH    = "outputs/best_model.pth"
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
 
# ── MODEL (Days 3-5) ────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, k, stride=stride,
                              padding=(k-1)//2, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        nn.init.kaiming_normal_(self.conv.weight, mode='fan_in',
                                nonlinearity='relu')
        nn.init.ones_(self.bn.weight); nn.init.zeros_(self.bn.bias)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))
 
class BrainTumourNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.block1 = ConvBlock(1,   32)
        self.block2 = ConvBlock(32,  64)
        self.block3 = ConvBlock(64,  128)
        self.block4 = ConvBlock(128, 256)
        self.pool   = nn.MaxPool2d(2, 2)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.mlp    = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(128, 64),  nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(64, num_classes))
    def forward(self, x):
        x = self.pool(self.block1(x))
        x = self.pool(self.block2(x))
        x = self.pool(self.block3(x))
        x = self.block4(x)
        x = self.gap(x).view(x.size(0), -1)
        return self.mlp(x)
 
 
# ── DATASET (Day 2) ─────────────────────────────────────
class BrainTumourDataset(Dataset):
    def __init__(self, root_dir, indices, transform=None):
        self.base      = ImageFolder(root=root_dir)
        self.indices   = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, label = self.base[self.indices[i]]
        if self.transform: img = self.transform(img)
        return img, label
 
 

In [3]:
# 1. EPOCH / BATCH / ITERATION ARITHMETIC
"""
batch      = N images processed together in one forward+backward
iteration  = one batch processed → ONE weight update
epoch      = one full pass through the training set
 
  iterations_per_epoch = floor(train_size / batch_size)
  total_updates        = iterations_per_epoch x epochs
 
Why batch at all (not one image at a time)?
  - GPU parallelism: 32 images cost barely more than 1
  - Gradient averaging over 32 samples = less noisy update direction
  - BatchNorm needs a batch to compute meaningful statistics
 
Why not the whole dataset at once (full-batch gradient descent)?
  - Won't fit in memory
  - Only ONE weight update per epoch → extremely slow convergence
  - The noise in mini-batch gradients actually HELPS escape
    poor local minima — this is the 'stochastic' in SGD
"""
CLASSES      = ["glioma", "meningioma", "notumour", "pituitary"]

base_ds  = ImageFolder(root=DATA_DIR)
labels   = base_ds.targets
all_idx  = list(range(len(base_ds)))
 
tv_idx, test_idx = train_test_split(all_idx, test_size=0.15,
                                    stratify=labels, random_state=42)
train_idx, val_idx = train_test_split(
    tv_idx, test_size=0.176,
    stratify=[labels[i] for i in tv_idx], random_state=42)
 
iters_per_epoch = len(train_idx) // BATCH_SIZE
print(f"Dataset:            {len(all_idx)} images, {len(CLASSES)} classes")
print(f"Train / Val / Test: {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")
print(f"Batch size:         {BATCH_SIZE}")
print(f"Iterations/epoch:   {len(train_idx)} // {BATCH_SIZE} = {iters_per_epoch}")
print(f"Total weight updates over {EPOCHS} epochs: "
      f"{iters_per_epoch} × {EPOCHS} = {iters_per_epoch*EPOCHS:,}")

Dataset:            1000 images, 4 classes
Train / Val / Test: 700 / 150 / 150
Batch size:         32
Iterations/epoch:   700 // 32 = 21
Total weight updates over 50 epochs: 21 × 50 = 1,050


In [4]:
# 2. GRADIENT ACCUMULATION — WHY zero_grad() IS MANDATORY
"""
PyTorch ACCUMULATES gradients by default:
  grad_new = grad_old + grad_from_this_backward
 
This is deliberate — it enables 'gradient accumulation' for
simulating large batches on small GPUs. But it means that if
you forget zero_grad(), batch 2's gradients ADD to batch 1's.
 
  Batch 1: grad = g1
  Batch 2: grad = g1 + g2     ← wrong, should be g2
  Batch 3: grad = g1 + g2 + g3 ← gradients grow unboundedly
  → weight updates become enormous → training diverges
"""
demo = BrainTumourNet().to(DEVICE); demo.train()
crit_demo = nn.CrossEntropyLoss()
x_d = torch.randn(4, 1, 128, 128, device=DEVICE)
y_d = torch.randint(0, 4, (4,), device=DEVICE)
 
print(f"{'':>28} {'grad norm':>12}")
print("-" * 43)
demo.zero_grad()
for i in range(1, 4):
    crit_demo(demo(x_d), y_d).backward()
    gn = demo.block1.conv.weight.grad.norm().item()
    print(f"  WITHOUT zero_grad, pass {i}: {gn:>12.6f}")
 
print()
for i in range(1, 4):
    demo.zero_grad()
    crit_demo(demo(x_d), y_d).backward()
    gn = demo.block1.conv.weight.grad.norm().item()
    print(f"  WITH zero_grad, pass {i}:    {gn:>12.6f}")
print("\n  → without zero_grad the norm grows every pass (accumulation)")
print("  → with zero_grad it stays constant (correct)")
demo.zero_grad()

                                grad norm
-------------------------------------------
  WITHOUT zero_grad, pass 1:     0.017261
  WITHOUT zero_grad, pass 2:     0.027942
  WITHOUT zero_grad, pass 3:     0.037360

  WITH zero_grad, pass 1:        0.015049
  WITH zero_grad, pass 2:        0.017079
  WITH zero_grad, pass 3:        0.018494

  → without zero_grad the norm grows every pass (accumulation)
  → with zero_grad it stays constant (correct)


In [5]:
# 3. WEIGHTED LOSS ACCUMULATION
"""
criterion() returns the MEAN loss over the batch.
If the last batch is smaller, its mean should count for less.
 
  WRONG:   total += loss.item()
           epoch_loss = total / num_batches
           → a 7-image batch weighted same as a 32-image batch
 
  CORRECT: total += loss.item() * images.size(0)
           epoch_loss = total / total_samples
           → true mean loss per sample
"""
batch_losses = [(32, 0.80), (32, 0.60), (32, 0.70), (7, 2.00)]
naive    = sum(l for _, l in batch_losses) / len(batch_losses)
weighted = sum(n*l for n, l in batch_losses) / sum(n for n, _ in batch_losses)
 
print(f"  Batches (size, mean_loss): {batch_losses}")
print(f"  Naive  (÷ num_batches):  {naive:.4f}   ← last batch over-weighted")
print(f"  Correct (÷ num_samples): {weighted:.4f}")
print(f"  Difference: {abs(naive-weighted):.4f} — matters when batches are uneven")

  Batches (size, mean_loss): [(32, 0.8), (32, 0.6), (32, 0.7), (7, 2.0)]
  Naive  (÷ num_batches):  1.0250   ← last batch over-weighted
  Correct (÷ num_samples): 0.7883
  Difference: 0.2367 — matters when batches are uneven


In [6]:
# 4. train_one_epoch() — THE CORE TRAINING FUNCTION
def train_one_epoch(model, loader, criterion, optimizer):
    """
    One full pass over the training set.
    model.train() → BatchNorm uses batch stats, Dropout is ON.
    Returns (mean_loss_per_sample, accuracy).
    """
    model.train()
    running_loss, correct, total = 0.0, 0, 0
 
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
 
        optimizer.zero_grad()                                   
        logits = model(images)                                  
        loss   = criterion(logits, targets)                     
        loss.backward()                                         
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
        optimizer.step()                                     
 
        running_loss += loss.item() * images.size(0)   # weighted accumulation
        correct      += (logits.argmax(1) == targets).sum().item()
        total        += targets.size(0)
 
    return running_loss / total, correct / total


In [7]:
# 5. evaluate() — VALIDATION WITH no_grad AND eval MODE
@torch.no_grad()   # disables computation graph — saves memory, faster
def evaluate(model, loader, criterion):
    """
    Evaluation pass — no weight updates.
    model.eval() → BatchNorm uses RUNNING stats, Dropout is OFF.
    Returns (mean_loss_per_sample, accuracy).
    """
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        logits = model(images)
        loss   = criterion(logits, targets)
 
        running_loss += loss.item() * images.size(0)
        correct      += (logits.argmax(1) == targets).sum().item()
        total        += targets.size(0)
 
    return running_loss / total, correct / total
 
print("""
train_one_epoch:  model.train()  + gradients ON  + weight updates
evaluate:         model.eval()   + gradients OFF + no updates
 
@torch.no_grad() on evaluate() does two things:
  - skips building the computation graph (≈2x faster)
  - halves memory usage (no stored intermediate activations)
It is SEPARATE from model.eval() — you need both.
""")

# ── DATA PIPELINE ───────────────────────────────────────
stat_tf = transforms.Compose([transforms.Grayscale(1),
                              transforms.Resize((IMG_SIZE, IMG_SIZE)),
                              transforms.ToTensor()])
stat_ds = ImageFolder(root=DATA_DIR, transform=stat_tf)
s, sq, n = 0.0, 0.0, 0
for i in train_idx:
    img, _ = stat_ds[i]
    s += img.sum().item(); sq += (img**2).sum().item(); n += img.numel()
MEAN = s / n
STD  = float(np.sqrt(sq/n - MEAN**2))
 
train_tf = transforms.Compose([
    transforms.Grayscale(1), transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5), transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize([MEAN], [STD])])
eval_tf = transforms.Compose([
    transforms.Grayscale(1), transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize([MEAN], [STD])])
 
train_loader = DataLoader(BrainTumourDataset(DATA_DIR, train_idx, train_tf),
                          BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(BrainTumourDataset(DATA_DIR, val_idx, eval_tf),
                          BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(BrainTumourDataset(DATA_DIR, test_idx, eval_tf),
                          BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
 
counts  = Counter(labels[i] for i in train_idx)
weights = torch.tensor([len(train_idx)/(len(CLASSES)*counts[c])
                        for c in range(len(CLASSES))], dtype=torch.float32)
 
print(f"Pixel stats (train only): mean={MEAN:.4f}, std={STD:.4f}")
print(f"Class weights: {[round(w.item(),3) for w in weights]}")
 


train_one_epoch:  model.train()  + gradients ON  + weight updates
evaluate:         model.eval()   + gradients OFF + no updates

@torch.no_grad() on evaluate() does two things:
  - skips building the computation graph (≈2x faster)
  - halves memory usage (no stored intermediate activations)
It is SEPARATE from model.eval() — you need both.



Pixel stats (train only): mean=0.4382, std=0.2221
Class weights: [0.833, 1.25, 1.0, 1.0]


In [8]:
# 6. CHECKPOINTING — WHAT TO SAVE AND WHY VAL LOSS
"""
Training for 50 epochs is pointless if we keep the LAST model.
The last epoch is usually overfitted; the best one is somewhere in
the middle. Checkpointing keeps the best and lets us recover it.

Select on VALIDATION loss, never training loss:
  train loss falls monotonically almost by definition — the model
  is directly optimising it. It says nothing about generalisation.
  Validation loss is the first thing to turn upward when the model
  starts memorising, which is exactly the moment we want to catch.

Why val LOSS rather than val ACCURACY:
  accuracy is a step function — it only moves when a prediction
  crosses the decision boundary, so it is noisy and plateaus.
  Loss reacts to confidence, so it detects degradation earlier.
  (We record both and report accuracy; we SELECT on loss.)

Save the state_dict, not the model object:
  torch.save(model) pickles the class definition by reference. Move
  the file, rename the class, or change the file layout and it will
  not load. A state_dict is just an OrderedDict of tensors.

We keep two copies:
  - an in-memory deepcopy for instant use after the loop
  - a file on disk so the run survives a kernel restart
"""
def save_checkpoint(model, optimizer, epoch, val_loss, val_acc,
                    path=CKPT_PATH):
    """Persist everything needed to resume or reproduce this exact model."""
    torch.save({
        "epoch":            epoch,
        "model_state":      model.state_dict(),
        "optimizer_state":  optimizer.state_dict(),   # Adam's m and v buffers
        "val_loss":         val_loss,
        "val_acc":          val_acc,
        "classes":          CLASSES,
        "img_size":         IMG_SIZE,
        "norm":             (MEAN, STD),   # test data must match train stats
    }, path)


def load_checkpoint(model, path=CKPT_PATH, optimizer=None):
    """Restore weights (and optionally optimiser state) in place."""
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    if optimizer is not None:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt


demo_model = BrainTumourNet().to(DEVICE)
demo_optim = torch.optim.Adam(demo_model.parameters(), lr=LR)
save_checkpoint(demo_model, demo_optim, epoch=0, val_loss=1.386, val_acc=0.25)

before = demo_model.block1.conv.weight.clone()
with torch.no_grad():                      # corrupt the weights
    demo_model.block1.conv.weight.mul_(0.0)
corrupted = demo_model.block1.conv.weight.clone()

ckpt = load_checkpoint(demo_model, optimizer=demo_optim)
after = demo_model.block1.conv.weight

print(f"Checkpoint saved to: {CKPT_PATH}")
print(f"  file size: {os.path.getsize(CKPT_PATH)/1e6:.2f} MB")
print(f"\nStored keys:")
for k, v in ckpt.items():
    kind = f"tensor dict ({len(v)} entries)" if k.endswith("_state") else v
    print(f"  {k:<18} {kind}")

print(f"\nRound-trip test:")
print(f"  weight norm before corruption : {before.norm().item():.6f}")
print(f"  weight norm after  corruption : {corrupted.norm().item():.6f}")
print(f"  weight norm after  restore    : {after.norm().item():.6f}")
print(f"  exact match after restore: {torch.equal(before, after)}")

Checkpoint saved to: outputs/best_model.pth
  file size: 1.73 MB

Stored keys:
  epoch              0
  model_state        tensor dict (30 entries)
  optimizer_state    tensor dict (2 entries)
  val_loss           1.386
  val_acc            0.25
  classes            ['glioma', 'meningioma', 'notumour', 'pituitary']
  img_size           128
  norm               (0.4382185150044305, 0.22207127062418935)

Round-trip test:
  weight norm before corruption : 7.660598
  weight norm after  corruption : 0.000000
  weight norm after  restore    : 7.660598
  exact match after restore: True


In [9]:
# 7. SCHEDULER PLACEMENT — PER-EPOCH, NOT PER-BATCH
"""
CosineAnnealingLR(T_max=50) is built to decay over 50 STEPS. We
call scheduler.step() once per epoch, so those 50 steps span the
whole 50-epoch run.

Put it inside the batch loop by mistake and it steps
iterations_per_epoch times MORE often. The cosine finishes its
entire schedule inside the first epoch or two, then keeps going —
the learning rate bottoms out almost immediately and the model
stops learning while appearing to train normally.

This is a silent failure: no error, no warning, just a model that
mysteriously underperforms. Worth demonstrating rather than
trusting yourself to remember.

Correct ordering within an epoch:
  for batch: zero_grad -> forward -> loss -> backward -> clip -> step
  after all batches: validate, then scheduler.step()
"""
def simulate(per_batch: bool, epochs=EPOCHS, iters=iters_per_epoch):
    m = nn.Linear(1, 1)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs,
                                                     eta_min=1e-6)
    trace = []
    for _ in range(epochs):
        for _ in range(iters):
            opt.step()
            if per_batch:
                sch.step()
        trace.append(opt.param_groups[0]["lr"])
        if not per_batch:
            sch.step()
    return trace

lr_epoch = simulate(per_batch=False)
lr_batch = simulate(per_batch=True)

print(f"T_max=50 epochs, {iters_per_epoch} iterations per epoch")
print(f"\n{'epoch':>6} {'correct (per-epoch)':>22} {'wrong (per-batch)':>20}")
print("-" * 50)
for e in (0, 1, 2, 4, 9, 24, 49):
    print(f"{e+1:>6} {lr_epoch[e]:>22.6e} {lr_batch[e]:>20.6e}")

print(f"\n  per-batch reaches eta_min after ~{EPOCHS}/{iters_per_epoch:.0f} "
      f"= {EPOCHS/iters_per_epoch:.1f} epochs")
print(f"  -> {(1 - lr_batch[2]/lr_epoch[2])*100:.1f}% of the intended learning "
      f"rate is already gone by epoch 3")

fig, ax = plt.subplots(figsize=(7, 4))
fig.patch.set_facecolor('#F8F8F6')
ax.plot(range(1, EPOCHS+1), lr_epoch, color='#1D9E75', lw=2.5,
        label='scheduler.step() per epoch  (correct)')
ax.plot(range(1, EPOCHS+1), lr_batch, color='#D85A30', lw=2.5,
        label='scheduler.step() per batch  (wrong)')
ax.set_yscale('log')
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning rate (log)")
ax.set_title("Scheduler placement changes the whole training run",
             fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/scheduler_placement.png", dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.close(fig)
print("\n  saved -> outputs/scheduler_placement.png")

T_max=50 epochs, 21 iterations per epoch

 epoch    correct (per-epoch)    wrong (per-batch)
--------------------------------------------------
     1           1.000000e-03         6.247206e-04
     2           9.990144e-04         6.278481e-05
     3           9.960613e-04         1.585687e-04
     5           9.843073e-04         9.755527e-04
    10           9.222418e-04         9.046040e-04
    25           5.318639e-04         5.005000e-04
    50           1.985649e-06         1.000000e-06

  per-batch reaches eta_min after ~50/21 = 2.4 epochs
  -> 84.1% of the intended learning rate is already gone by epoch 3

  saved -> outputs/scheduler_placement.png


In [10]:
# 8. FULL TRAINING RUN — EVERYTHING ASSEMBLED
"""
Every component from Days 2-6 in one loop:

  Day 2  train_loader / val_loader with train-only augmentation
  Day 3  ConvBlock
  Day 4  BrainTumourCNN feature extractor
  Day 5  MLP head + dropout
  Day 6  weighted CrossEntropy, Adam, cosine schedule, grad clipping
  Day 7  epoch loop, weighted accumulation, checkpointing

The per-epoch order is fixed and each position matters:
  1. train_one_epoch   weights update, dropout on, BN uses batch stats
  2. evaluate          no grad, dropout off, BN uses running stats
  3. checkpoint        only if val_loss improved
  4. scheduler.step    exactly once per epoch (section 7)
"""
model     = BrainTumourNet().to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights.to(DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=LR,
                             weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6)

history   = {"train_loss": [], "val_loss": [],
             "train_acc": [],  "val_acc": [], "lr": []}
best_loss = float("inf")
best_state, best_epoch = None, 0
start = time.time()

print(f"Device: {DEVICE}  |  {sum(p.numel() for p in model.parameters()):,} params")
print(f"Training {len(train_idx)} images for {EPOCHS} epochs "
      f"({iters_per_epoch} iterations/epoch)\n")
print(f"{'epoch':>5} {'train loss':>11} {'train acc':>10} "
      f"{'val loss':>10} {'val acc':>9} {'lr':>9}  {'':<4}")
print("-" * 62)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    lr_now = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss);   history["val_acc"].append(va_acc)
    history["lr"].append(lr_now)

    flag = ""
    if va_loss < best_loss:                       # select on val LOSS
        best_loss, best_epoch = va_loss, epoch
        best_state = copy.deepcopy(model.state_dict())
        save_checkpoint(model, optimizer, epoch, va_loss, va_acc)
        flag = "*"

    scheduler.step()                              # once per epoch

    if epoch <= 5 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f"{epoch:>5} {tr_loss:>11.4f} {tr_acc:>10.4f} "
              f"{va_loss:>10.4f} {va_acc:>9.4f} {lr_now:>9.2e}  {flag:<4}")

model.load_state_dict(best_state)                 # restore the best epoch

print("-" * 62)
print(f"  * = new best validation loss (checkpoint written)")
print(f"\nFinished in {time.time()-start:.1f}s")
print(f"Best epoch: {best_epoch}/{EPOCHS}  "
      f"val_loss={best_loss:.4f}  val_acc={history['val_acc'][best_epoch-1]:.4f}")
print(f"Final epoch val_acc={history['val_acc'][-1]:.4f} "
      f"-> best is {'later' if best_epoch == EPOCHS else 'NOT the last epoch'}")

Device: cuda  |  429,732 params
Training 700 images for 50 epochs (21 iterations/epoch)

epoch  train loss  train acc   val loss   val acc        lr      
--------------------------------------------------------------
    1      1.3853     0.2411     1.3800    0.2000  1.00e-03  *   
    2      1.3528     0.3542     1.3218    0.3000  9.99e-04  *   
    3      1.2308     0.4315     1.0074    0.5933  9.96e-04  *   
    4      0.9490     0.5476     0.7472    0.6267  9.91e-04  *   
    5      0.7677     0.6220     1.4413    0.2133  9.84e-04      
   10      0.4513     0.8080     0.3148    0.8933  9.22e-04  *   
   15      0.4236     0.8125     0.2371    0.9067  8.19e-04  *   
   20      0.2673     0.9033     0.1314    0.9667  6.84e-04  *   
   25      0.1679     0.9301     0.0787    0.9800  5.32e-04  *   
   30      0.1172     0.9628     0.0451    0.9867  3.76e-04      
   35      0.1094     0.9628     0.0806    0.9800  2.33e-04      
   40      0.0806     0.9747     0.0149    0.9933  1.16e

In [11]:
# 9. LOSS CURVE INTERPRETATION — READING OVERFITTING
"""
The gap between the training and validation curves is the single
most informative plot in the whole project.

  both still falling            -> underfitting, train longer
  train falls, val flattens     -> capacity reached
  train falls, val RISES        -> overfitting; the model is
                                   memorising individual scans
  val below train               -> normal here, not a bug:
                                   dropout and augmentation only
                                   apply during training, so the
                                   training pass is handicapped

The generalisation gap (val_loss - train_loss) is what Day 8's
regularisation and augmentation work is meant to shrink.
"""
ep = range(1, EPOCHS + 1)
gap = [v - t for v, t in zip(history["val_loss"], history["train_loss"])]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#F8F8F6')

axes[0].plot(ep, history["train_loss"], color='#378ADD', lw=2, label='train')
axes[0].plot(ep, history["val_loss"],   color='#D85A30', lw=2, label='val')
axes[0].axvline(best_epoch, color='#1D9E75', ls='--', lw=1.5,
                label=f'best (epoch {best_epoch})')
axes[0].set_title("Loss", fontsize=10, fontweight='bold')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-entropy")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history["train_acc"], color='#378ADD', lw=2, label='train')
axes[1].plot(ep, history["val_acc"],   color='#D85A30', lw=2, label='val')
axes[1].axvline(best_epoch, color='#1D9E75', ls='--', lw=1.5)
axes[1].axhline(0.25, color='gray', ls=':', lw=1.5, label='chance (4 classes)')
axes[1].set_title("Accuracy", fontsize=10, fontweight='bold')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.02)
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

axes[2].plot(ep, gap, color='#9B59B6', lw=2)
axes[2].axhline(0, color='gray', ls=':', lw=1.5)
axes[2].fill_between(ep, 0, gap, color='#9B59B6', alpha=0.2)
axes[2].set_title("Generalisation gap\n(val loss - train loss)",
                  fontsize=10, fontweight='bold')
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Gap")
axes[2].grid(alpha=0.3)

plt.suptitle("Day 7 — training dynamics", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/training_curves.png", dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.close(fig)

print(f"  saved -> outputs/training_curves.png\n")
print(f"{'':>16} {'first':>9} {'best':>9} {'last':>9}")
print("-" * 47)
for key in ("train_loss", "val_loss", "train_acc", "val_acc"):
    h = history[key]
    print(f"{key:>16} {h[0]:>9.4f} {h[best_epoch-1]:>9.4f} {h[-1]:>9.4f}")

print(f"\nGeneralisation gap at best epoch: {gap[best_epoch-1]:+.4f}")
print(f"Generalisation gap at final epoch: {gap[-1]:+.4f}")
if gap[-1] > gap[best_epoch-1] + 0.05:
    print("  -> gap widened after the best epoch: overfitting began.")
    print("     This is exactly what checkpoint-on-val-loss protects against.")
else:
    print("  -> gap stayed flat: the model has not started memorising yet.")

  saved -> outputs/training_curves.png

                     first      best      last
-----------------------------------------------
      train_loss    1.3853    0.0702    0.0695
        val_loss    1.3800    0.0053    0.0088
       train_acc    0.2411    0.9762    0.9792
         val_acc    0.2000    1.0000    1.0000

Generalisation gap at best epoch: -0.0649
Generalisation gap at final epoch: -0.0607
  -> gap stayed flat: the model has not started memorising yet.


In [12]:
# 10. FINAL TEST EVALUATION + VERIFICATION CHECKLIST
"""
The test set has been untouched until now. That is the whole point
— it was never used for weight updates (train) or for choosing the
checkpoint (val), so it is the only unbiased estimate we have.

Report it ONCE. Tuning anything in response to the test number
turns it into a second validation set and the estimate is no longer
honest. Day 9 expands this into the confusion matrix and per-class
precision/recall for the report.

We reload from disk rather than using the in-memory model, which
also proves the checkpoint file is complete and usable on its own.

READ THE NUMBER BELOW WITH SUSPICION.
  data/brain_tumour currently holds SYNTHETIC images generated by
  make_synthetic_mri() in Day 2, not real scans. Each class is drawn
  from a different, cleanly separable distribution, so near-100%
  accuracy is expected and means nothing clinically. It confirms the
  training machinery works end to end — that is all it confirms.

  On the real Kaggle MRI data expect roughly 90-97%, a visible
  generalisation gap, and classes that genuinely confuse each other
  (glioma vs meningioma especially). Point DATA_DIR at the real
  dataset before quoting any figure in the report.
"""
final_model = BrainTumourNet().to(DEVICE)
ckpt = load_checkpoint(final_model, CKPT_PATH)

test_loss, test_acc = evaluate(final_model, test_loader, criterion)

print(f"Loaded checkpoint from epoch {ckpt['epoch']} "
      f"(val_loss={ckpt['val_loss']:.4f})")
print(f"\n{'split':>8} {'loss':>9} {'accuracy':>10}")
print("-" * 30)
print(f"{'train':>8} {history['train_loss'][best_epoch-1]:>9.4f} "
      f"{history['train_acc'][best_epoch-1]:>10.4f}")
print(f"{'val':>8} {history['val_loss'][best_epoch-1]:>9.4f} "
      f"{history['val_acc'][best_epoch-1]:>10.4f}")
print(f"{'test':>8} {test_loss:>9.4f} {test_acc:>10.4f}   <- report this one")

final_model.eval()
correct = torch.zeros(len(CLASSES))
totals  = torch.zeros(len(CLASSES))
with torch.no_grad():
    for images, targets in test_loader:
        preds = final_model(images.to(DEVICE)).argmax(1).cpu()
        for c in range(len(CLASSES)):
            mask = targets == c
            totals[c]  += mask.sum()
            correct[c] += (preds[mask] == c).sum()

print(f"\nPer-class test accuracy:")
for c, name in enumerate(CLASSES):
    acc = (correct[c] / totals[c]).item() if totals[c] > 0 else float('nan')
    bar = "#" * int(acc * 30)
    print(f"  {name:<12} {int(correct[c]):>3}/{int(totals[c]):<3} "
          f"{acc:>6.3f}  {bar}")

print("\n" + "=" * 60)
print("DAY 7 VERIFICATION CHECKLIST")
print("=" * 60)
final_checks = [
    ("training ran for all epochs",
     len(history["train_loss"]) == EPOCHS),
    ("loss decreased from epoch 1",
     history["train_loss"][-1] < history["train_loss"][0]),
    ("model beats 25% chance on test",
     test_acc > 0.25),
    ("checkpoint file exists on disk",
     os.path.exists(CKPT_PATH)),
    ("checkpoint reloads and evaluates",
     test_loss > 0),
    ("best epoch chosen on val loss",
     abs(min(history["val_loss"]) - best_loss) < 1e-9),
    ("scheduler completed its cosine",
     history["lr"][-1] < history["lr"][0]),
    ("every class predicted at least once",
     bool((correct > 0).all())),
]
for label, ok in final_checks:
    print(f"  {'OK  ' if ok else 'FAIL'}  {label}")

g_best = gap[best_epoch-1]
print(f"\n  Generalisation gap at the selected epoch: {g_best:+.3f}")
if g_best < 0:
    print("  Negative because dropout and augmentation handicap the training")
    print("  pass only — the val pass sees clean images with dropout off.")
print(f"\n  NOTE: this run used SYNTHETIC data (Day 2). Re-run against the")
print(f"  real Kaggle dataset before using any of these numbers.")
print(f"  Day 8 next: regularisation and augmentation on real data.")

Loaded checkpoint from epoch 43 (val_loss=0.0053)

   split      loss   accuracy
------------------------------
   train    0.0702     0.9762
     val    0.0053     1.0000
    test    0.0221     0.9867   <- report this one

Per-class test accuracy:
  glioma        43/45   0.956  ############################
  meningioma    30/30   1.000  ##############################
  notumour      38/38   1.000  ##############################
  pituitary     37/37   1.000  ##############################

DAY 7 VERIFICATION CHECKLIST
  OK    training ran for all epochs
  OK    loss decreased from epoch 1
  OK    model beats 25% chance on test
  OK    checkpoint file exists on disk
  OK    checkpoint reloads and evaluates
  OK    best epoch chosen on val loss
  OK    scheduler completed its cosine
  OK    every class predicted at least once

  Generalisation gap at the selected epoch: -0.065
  Negative because dropout and augmentation handicap the training
  pass only — the val pass sees clean images 